In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -r "/content/drive/MyDrive/GEO_AI_Cropland_Mapping_Challenge/requirements.txt"

In [ ]:
import ee
import geopandas as gpd
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import time
import geemap
import os
import random
import warnings

warnings.filterwarnings("ignore")

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
start = time.time()

In [ ]:
DATA_PATH = "/content/drive/MyDrive/GEO_AI_Cropland_Mapping_Challenge"
TEST_PATH = DATA_PATH + "/" + "raw_test_data/"
S1_TEST = "Sentinel1.csv"
S2_TEST = "Sentinel2.csv"

In [ ]:
#read the test data inorder to match columns that will be retrieved for train data
s1_test = pd.read_csv(TEST_PATH + S1_TEST)
s2_test = pd.read_csv(TEST_PATH + S2_TEST)
print("S1 : ",s1_test.columns)
print("S2 : ",s2_test.columns)

S1 :  Index(['ID', 'VH', 'VV', 'date', 'orbit', 'polarization', 'rel_orbit',
       'translated_lat', 'translated_lon'],
      dtype='object')
S2 :  Index(['B11', 'B12', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'ID',
       'cloud_pct', 'date', 'solar_azimuth', 'solar_zenith', 'translated_lat',
       'translated_lon'],
      dtype='object')


In [ ]:
s1_test.head()

,ID,VH,VV,date,orbit,polarization,rel_orbit,translated_lat,translated_lon
0,ID_AFQOFP,-21.479683,-16.633259,2022-07-01,DESCENDING,"[VV, VH]",78.0,41.652292,72.144256
1,ID_AFQOFP,-24.769110,-15.943674,2022-07-01,DESCENDING,"[VV, VH]",78.0,41.652289,72.144375
2,ID_AFQOFP,-25.370838,-15.185609,2022-07-01,DESCENDING,"[VV, VH]",78.0,41.652286,72.144495
3,ID_AFQOFP,-24.134005,-16.351102,2022-07-01,DESCENDING,"[VV, VH]",78.0,41.652283,72.144614
4,ID_AFQOFP,-20.654249,-16.792723,2022-07-01,DESCENDING,"[VV, VH]",78.0,41.652280,72.144733


In [ ]:
print("unique orbit : ",s1_test['orbit'].unique())
print("unique polarization : ",s1_test['polarization'].unique())
print("unique rel_orbit : ",s1_test['rel_orbit'].unique())
print("unique _dates : ",s1_test['date'].unique())

unique orbit :  ['DESCENDING']
unique polarization :  ['[VV, VH]']
unique rel_orbit :  [ 78.   5.  35. 108. 137.]
unique _dates :  ['2022-07-01' '2022-07-08' '2022-07-13' '2022-07-20' '2022-07-25'
 '2022-08-01' '2022-08-06' '2022-08-13' '2022-08-18' '2022-08-25'
 '2022-08-30' '2022-09-06' '2022-09-11' '2022-09-18' '2022-09-23'
 '2022-09-30' '2022-10-05' '2022-10-12' '2022-10-17' '2022-10-24'
 '2022-10-29' '2022-11-05' '2022-11-10' '2022-11-17' '2022-11-22'
 '2022-11-29' '2022-12-04' '2022-12-11' '2022-12-16' '2022-12-23'
 '2022-12-28' '2023-01-04' '2023-01-09' '2023-01-16' '2023-01-21'
 '2023-01-28' '2023-02-02' '2023-02-09' '2023-02-14' '2023-02-21'
 '2023-02-26' '2023-03-05' '2023-03-10' '2023-03-17' '2023-03-22'
 '2023-03-29' '2023-04-03' '2023-04-10' '2023-04-15' '2023-04-22'
 '2023-04-27' '2023-05-04' '2023-05-09' '2023-05-16' '2023-05-21'
 '2023-05-28' '2023-06-02' '2023-06-09' '2023-06-14' '2023-06-21'
 '2021-07-01' '2021-07-06' '2021-07-13' '2021-07-18' '2021-07-30'
 '2021-08-0

In [ ]:
#authenticate connection to google earth engine
# If you’ve never authenticated in this notebook:
ee.Authenticate(auth_mode  = "notebook")

To authorize access needed by Earth Engine, open the following URL in a web browser and follow the instructions. If the web browser does not start automatically, please manually browse the URL below.

    https://code.earthengine.google.com/client-auth?scopes=https%3A//www.googleapis.com/auth/earthengine%20https%3A//www.googleapis.com/auth/cloud-platform%20https%3A//www.googleapis.com/auth/drive%20https%3A//www.googleapis.com/auth/devstorage.full_control&request_id=DC_0F-cCUnaZgoz_I2HsLOrx4O-F7luyGYBU5kCLz7Y&tc=YcfjSzl33hpFtNZG0SJZI05WXcnf2TWT17l_zTHcrn0&cc=uaAbH0E6X0o5ysck3lci-PMtD4VkhNCCKlZ9eZwlv1k

The authorization workflow will generate a code, which you should paste in the box below.
Enter verification code: 4/1AVGzR1CZRXhQcyx9yuxUJ1J502y8YyL28NPVYCibUwyYEoSbz2qx-aEz-_s

Successfully saved authorization token.


To authorize access needed by Earth Engine, open the following URL in a web browser and follow the instructions:

https://code.earthengine.google.com/client-auth?scopes=https%3A//www.googleapis.com/auth/earthengine%20https%3A//www.googleapis.com/auth/devstorage.full_control&request_id=mrhUXGxXj7vySDzsLIWVG68BevHt0sL3neKmSDxwVWI&tc=mEEfgQF9Op6K2zqvwJyNRGGDlTT0wEvWB1-yoULkQNg&cc=rYxd8E9jOurg7jFK2lebJ5ovgBhHEUZjV2crVQhdBY4

In [ ]:
#initialize earth engine project
ee.Initialize(project=<enter your earth engine project name>)

In [ ]:
#si data collection database
s1  = ee.ImageCollection('COPERNICUS/S1_GRD')

In [ ]:
#load the train geolocation files
TRAIN_DATA_PATH =  DATA_PATH + "/" + "raw_train_data/"
FERGANA_DATA_PATH = TRAIN_DATA_PATH + "Fergana_training_samples.shp"
ORENBURG_DATA_PATH = TRAIN_DATA_PATH + "Orenburg_training_samples.shp"

In [ ]:
START, END = "2022-01-01", "2022-12-31"   # date window matching dataset
S1_BANDS = ['ID', 'VH', 'VV', 'date', 'orbit', 'polarization', 'rel_orbit']
S2_BANDS = ['B11', 'B12', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'ID',
       'cloud_pct', 'date', 'solar_azimuth', 'solar_zenith']


# SENTINEL 1

In [ ]:
ORBIT_PASS  = 'DESCENDING'
REL_ORBITS  = [5, 35, 78, 108, 137]
REQUIRED_POLS = ['VV','VH']
PAD_S1      = 1                 # ± days around each target date
DATES_PER_EXPORT = 8            # small chunks = safer
DRIVE_FOLDER = "EE_S1_Exports"   # Drive folder (auto-created if missing) ,watch out if there are any other folders in the drive with similar name
FILENAME_PREFIX = "train_S1"     # prefix for exported CSVs

In [ ]:
dates_s1 = s1_test['date'].unique()
print(f"[INFO] Found {len(dates_s1)} unique S1 dates from test.")

[INFO] Found 294 unique S1 dates from test.


In [ ]:
#load the data in chunks to ensure minimal memory usage
def chunk(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

#loading the geolocation coordinates from the shp files
def load_points(shp_path, region_name):
    gdf = gpd.read_file(shp_path)
    if gdf.crs is None:
        print(f"[WARN] No CRS in {shp_path}. Assuming EPSG:4326.")
        gdf = gdf.set_crs(epsg=4326)
    else:
        gdf = gdf.to_crs(epsg=4326)
    assert "ID" in gdf.columns, f"{shp_path} must contain an 'ID' column."

    feats = []
    for _, r in gdf.iterrows():
        geom = ee.Geometry.Point([r.geometry.x, r.geometry.y])
        props = {"ID": r["ID"], "region": region_name}
        feats.append(ee.Feature(geom, props))
    fc = ee.FeatureCollection(feats)
    return fc, gdf  # (FeatureCollection, GeoDataFrame)

#tiem window per load
def window(d_iso, pad_days):
    d0 = datetime.fromisoformat(d_iso)
    beg = (d0 - timedelta(days=pad_days)).strftime("%Y-%m-%d")
    end = (d0 + timedelta(days=pad_days+1)).strftime("%Y-%m-%d")
    return beg, end

def s1_collection_filtered(aoi, beg, end):
    # Enforce IW, DESCENDING, both VV & VH, and rel_orbit in the given set
    col = (ee.ImageCollection("COPERNICUS/S1_GRD")
           .filterBounds(aoi)
           .filterDate(beg, end)
           .filter(ee.Filter.eq('instrumentMode', 'IW'))
           .filter(ee.Filter.eq('orbitProperties_pass', ORBIT_PASS))
           .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
           .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
           .filter(ee.Filter.inList('relativeOrbitNumber_start', REL_ORBITS)))
    return col

def empty_s1_placeholder():
    # Fully masked VV & VH bands so sampling yields nulls instead of throwing
    vv = ee.Image.constant(0).updateMask(ee.Image.constant(0)).rename('VV')
    vh = ee.Image.constant(0).updateMask(ee.Image.constant(0)).rename('VH')
    return vv.addBands(vh)

def s1_image_for_date(aoi, d_iso):
    beg, end = window(d_iso, PAD_S1)
    col = s1_collection_filtered(aoi, beg, end)
    img = ee.Image(ee.Algorithms.If(col.size().gt(0), col.select(REQUIRED_POLS).median(), empty_s1_placeholder()))
    return img


In [ ]:
def s1_meta_for_date(aoi, d_iso):
    beg, end = window(d_iso, PAD_S1)
    col = s1_collection_filtered(aoi, beg, end)
    one = col.first()

    # Safely coerce polarization (list) to a bracketed string like "[VV, VH]"
    pol_list = ee.Algorithms.If(
        one,
        ee.List(ee.Image(one).get('transmitterReceiverPolarisation')),
        ee.List([])
    )
    pol_str = ee.String('[').cat(ee.List(pol_list).join(', ')).cat(']')

    return {
        'orbit':        ee.Algorithms.If(one, ee.String(ee.Image(one).get('orbitProperties_pass')), None),
        'polarization': ee.Algorithms.If(one, pol_str, None),
        'rel_orbit':    ee.Algorithms.If(one, ee.Number(ee.Image(one).get('relativeOrbitNumber_start')), None),
    }

In [ ]:
def build_fc_for_dates(fc_points, aoi, dates):
    """Merge samples for a small list of dates into one FeatureCollection."""
    out = ee.FeatureCollection([])
    for d in dates:
        img  = s1_image_for_date(aoi, d)
        meta = s1_meta_for_date(aoi, d)
        samp = (img.sampleRegions(collection=fc_points,
                                  properties=['ID','region'],
                                  scale=10,
                                  geometries=False)
                  .map(lambda f: f.set({
                      'date': d,
                      'orbit': meta['orbit'],
                      'polarization': meta['polarization'],
                      'rel_orbit': meta['rel_orbit']
                  })))
        out = out.merge(samp)
    return out

In [ ]:
#export the data to google drive
def export_fc_to_drive(fc, description, filename_prefix, folder, selectors):
    task = ee.batch.Export.table.toDrive(
        collection = fc,
        description = description,
        folder = folder,
        fileNamePrefix = filename_prefix,
        fileFormat = 'CSV',
        selectors = selectors  # enforce column order/subset
    )
    task.start()
    print(f"[TASK] Started: {description} → Drive/{folder}/{filename_prefix}.csv")

# -------- main -------------
dates_all = s1_test['date'].unique()
print(f"[INFO] Unique S1 dates from test: {len(dates_all)}")

[INFO] Unique S1 dates from test: 294


In [ ]:
%%time
# Load region point sets
fc_ferg, gdf_ferg = load_points(FERGANA_DATA_PATH,  "Fergana")
fc_oren, gdf_oren = load_points(ORENBURG_DATA_PATH, "Orenburg")

# AOIs (small buffer to ensure coverage near edges)
aoi_ferg = fc_ferg.geometry().buffer(5000)
aoi_oren = fc_oren.geometry().buffer(5000)

# Column order to match your test schema
SELECTORS = ['ID','VH','VV','date','orbit','polarization','rel_orbit','region']

# Create chunked exports per region
import time

# helper STARTS the task and prints a message; it returns nothing
# def export_fc_to_drive(...): ... task.start(); print(...)

for region_name, fc_pts, aoi in [
    ("Fergana",  fc_ferg, aoi_ferg),
    ("Orenburg", fc_oren, aoi_oren),
]:
    for i, dates_chunk in enumerate(chunk(dates_all, DATES_PER_EXPORT), start=1):
        fc_chunk = build_fc_for_dates(fc_pts, aoi, dates_chunk)
        desc  = f"S1_{region_name}_chunk_{i:03d}"
        fname = f"{FILENAME_PREFIX}_{region_name}_chunk_{i:03d}"

        # Starts immediately inside the helper
        export_fc_to_drive(fc_chunk, desc, fname, DRIVE_FOLDER, SELECTORS)

        # Find the just-started task by description and poll it
        task = next((t for t in ee.batch.Task.list() if t.config.get('description') == desc), None)
        if task is None:
            print(f"[WARN] Could not find task just started: {desc}")
            continue

        while task.active():
            print(f"[{desc}] running...")
            time.sleep(30)

        print(f"[{desc}] state:", task.status().get('state'))

print("\n[INFO] Export tasks submitted.")
print("Tip: keep chunks small (e.g., 5–10 dates) if you still hit memory limits.")

[WARN] No CRS in /content/drive/MyDrive/4th_place_GEO_AI_Cropland_Mapping_Challenge/raw_train_data/Fergana_training_samples.shp. Assuming EPSG:4326.
[WARN] No CRS in /content/drive/MyDrive/4th_place_GEO_AI_Cropland_Mapping_Challenge/raw_train_data/Orenburg_training_samples.shp. Assuming EPSG:4326.
[TASK] Started: S1_Fergana_chunk_001 → Drive/EE_S1_Exports/train_S1_Fergana_chunk_001.csv
[S1_Fergana_chunk_001] running...
[S1_Fergana_chunk_001] running...
[S1_Fergana_chunk_001] state: COMPLETED
[TASK] Started: S1_Fergana_chunk_002 → Drive/EE_S1_Exports/train_S1_Fergana_chunk_002.csv
[S1_Fergana_chunk_002] running...
[S1_Fergana_chunk_002] running...
[S1_Fergana_chunk_002] running...
[S1_Fergana_chunk_002] state: COMPLETED
[TASK] Started: S1_Fergana_chunk_003 → Drive/EE_S1_Exports/train_S1_Fergana_chunk_003.csv
[S1_Fergana_chunk_003] running...
[S1_Fergana_chunk_003] running...
[S1_Fergana_chunk_003] state: COMPLETED
[TASK] Started: S1_Fergana_chunk_004 → Drive/EE_S1_Exports/train_S1_Ferga

In [ ]:
s1_data_time = time.time()
print("submitted S1 data for download in : ", (s1_data_time-start)/60 ,"minutes")

submitted S1 data for download in :  83.59717988570532 minutes


# SENTINEL 2

In [ ]:
s2_test.head()

,B11,B12,B2,B3,B4,B5,B6,B7,B8,B8A,ID,cloud_pct,date,solar_azimuth,solar_zenith,translated_lat,translated_lon
0,2169,1820,1328,1610,1670,1985,2446,2628,2598,2638,ID_ZHZRHO,6.980395,2021-07-05,139.093139,22.625533,40.935173,71.617062
1,2151,1770,1306,1586,1640,1961,2495,2691,2684,2732,ID_ZHZRHO,6.980395,2021-07-05,139.093139,22.625533,40.935171,71.617180
2,2169,1820,1456,1674,1808,1985,2446,2628,2486,2638,ID_ZHZRHO,6.980395,2021-07-05,139.093139,22.625533,40.935085,71.616940
3,2169,1820,1284,1604,1658,1985,2446,2628,2658,2638,ID_ZHZRHO,6.980395,2021-07-05,139.093139,22.625533,40.935083,71.617059
4,2151,1770,1242,1522,1564,1961,2495,2691,2696,2732,ID_ZHZRHO,6.980395,2021-07-05,139.093139,22.625533,40.935081,71.617177


In [ ]:
print("max cloud pct : ",s2_test['cloud_pct'].max())
print("unique dates : ",s2_test['date'].unique())

max cloud pct :  19.97716
unique dates :  ['2021-07-05' '2021-07-07' '2021-07-17' '2021-07-20' '2021-07-22'
 '2021-07-25' '2021-07-27' '2021-07-30' '2021-08-01' '2021-08-04'
 '2021-08-06' '2021-08-11' '2021-08-14' '2021-08-16' '2021-08-19'
 '2021-08-21' '2021-08-24' '2021-08-26' '2021-08-29' '2021-08-31'
 '2021-09-03' '2021-09-05' '2021-09-08' '2021-09-13' '2021-09-15'
 '2021-09-18' '2021-09-20' '2021-09-23' '2021-09-25' '2021-09-28'
 '2021-09-30' '2021-10-03' '2021-10-13' '2021-10-15' '2021-10-18'
 '2021-10-20' '2021-10-23' '2021-10-25' '2021-10-30' '2021-11-02'
 '2021-11-09' '2021-11-14' '2021-11-19' '2021-11-22' '2021-11-24'
 '2021-11-29' '2021-12-02' '2021-12-09' '2021-12-12' '2021-12-17'
 '2021-12-24' '2021-12-27' '2021-12-29' '2022-01-08' '2022-01-11'
 '2022-01-21' '2022-01-26' '2022-01-31' '2022-02-05' '2022-02-10'
 '2022-02-15' '2022-02-17' '2022-02-20' '2022-02-22' '2022-02-27'
 '2022-03-02' '2022-03-04' '2022-04-01' '2022-04-03' '2022-04-06'
 '2022-04-08' '2022-04-13' '2022-0

In [ ]:
TEST_S2_CSV        = "data/test_S2.csv"   # must have a 'date' column

PAD_S2             = 2        # ± days around each target date (start with 3; increase only if coverage is low)
DATES_PER_EXPORT   = 8        # smaller chunks = safer server memory
DRIVE_FOLDER       = "EE_S2_Exports"    # will appear in your Google Drive. download to local
FILENAME_PREFIX    = "train_S2"         # file name prefix for exports

In [ ]:
# Bands to export:
S2_BANDS = ['B11','B12','B2','B3','B4','B5','B6','B7','B8','B8A']
SELECTORS = S2_BANDS + ['ID','cloud_pct','date','solar_azimuth','solar_zenith','region']

def chunk(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

def load_points(shp_path, region_name):
    gdf = gpd.read_file(shp_path)
    if gdf.crs is None:
        print(f"[WARN] No CRS in {shp_path}. Assuming EPSG:4326.")
        gdf = gdf.set_crs(epsg=4326)
    else:
        gdf = gdf.to_crs(epsg=4326)
    assert "ID" in gdf.columns, f"{shp_path} must contain an 'ID' column."

    feats = []
    for _, r in gdf.iterrows():
        geom = ee.Geometry.Point([r.geometry.x, r.geometry.y])
        props = {"ID": r["ID"], "region": region_name}
        feats.append(ee.Feature(geom, props))
    fc = ee.FeatureCollection(feats)
    return fc, gdf

def window(d_iso, pad_days):
    d0 = datetime.fromisoformat(d_iso)
    beg = (d0 - timedelta(days=pad_days)).strftime("%Y-%m-%d")
    end = (d0 + timedelta(days=pad_days+1)).strftime("%Y-%m-%d")
    return beg, end

def mask_s2_clouds(img):
    # QA60 bits 10 & 11 are clouds/cirrus
    qa = img.select('QA60')
    cloud_bits = (1 << 10) | (1 << 11)
    mask = qa.bitwiseAnd(cloud_bits).eq(0)
    return img.updateMask(mask)

def make_nan_image():
    # Create an image of NaN for numeric placeholders
    return ee.Image(0).divide(0)  # yields NaN

def s2_image_for_date(aoi, d_iso):
    beg, end = window(d_iso, PAD_S2)
    col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
           .filterBounds(aoi)
           .filterDate(beg, end)
           .map(mask_s2_clouds))

    # pick the least cloudy by CLOUDY_PIXEL_PERCENTAGE property.
    # Attach cloud_pct / solar angles as constant bands.
    def attach_props(im):
        cloud_pct  = ee.Image.constant(im.get('CLOUDY_PIXEL_PERCENTAGE')).rename('cloud_pct')
        sol_az     = ee.Image.constant(im.get('MEAN_SOLAR_AZIMUTH_ANGLE')).rename('solar_azimuth')
        sol_zen    = ee.Image.constant(im.get('MEAN_SOLAR_ZENITH_ANGLE')).rename('solar_zenith')
        return im.addBands([cloud_pct, sol_az, sol_zen])

    col_with_props = col.map(attach_props)
    has_img = col_with_props.size().gt(0)

    # Build image or placeholder:
    def _build_from_collection():
        best = col_with_props.sort('CLOUDY_PIXEL_PERCENTAGE').first()
        return ee.Image(best).select(S2_BANDS + ['cloud_pct','solar_azimuth','solar_zenith'])

    def _placeholder():
        # Masked bands + NaN metadata bands so sampling yields NaNs (not errors)
        empty_bands = [ee.Image.constant(0).updateMask(0).rename(b) for b in S2_BANDS]
        nan_img = make_nan_image()
        props_bands = [nan_img.rename('cloud_pct'),
                       nan_img.rename('solar_azimuth'),
                       nan_img.rename('solar_zenith')]
        return ee.Image.cat(empty_bands + props_bands)

    return ee.Image(ee.Algorithms.If(has_img, _build_from_collection(), _placeholder()))

def build_fc_for_dates(fc_points, aoi, dates, region_name):
  #create the data strcture for the dates specified
    out = ee.FeatureCollection([])
    for d in dates:
        img = s2_image_for_date(aoi, d)
        samp = (img.sampleRegions(collection=fc_points,
                                  properties=['ID','region'],
                                  scale=10,
                                  geometries=False)
                  .map(lambda f: f.set({'date': d})))
        out = out.merge(samp)
    return out

In [ ]:
# helper STARTS the task and prints a message; it returns nothing
# def export_fc_to_drive(...): ... task.start(); print(...)

def export_fc_to_drive(fc, description, filename_prefix, folder, selectors):
    task = ee.batch.Export.table.toDrive(
        collection = fc,
        description = description,
        folder = folder,
        fileNamePrefix = filename_prefix,
        fileFormat = 'CSV',
        selectors = selectors
    )
    task.start()
    print(f"[TASK] Started: {description} → Drive/{folder}/{filename_prefix}.csv")

# -------- main -------------
dates_all = s2_test['date'].unique()
print(f"[INFO] Unique S2 dates from test: {len(dates_all)}")

# Load point sets
fc_ferg, _ = load_points(FERGANA_DATA_PATH,  "Fergana")
fc_oren, _ = load_points(ORENBURG_DATA_PATH, "Orenburg")

# AOIs with modest buffer
aoi_ferg = fc_ferg.geometry().buffer(5000)
aoi_oren = fc_oren.geometry().buffer(5000)

for region_name, fc_pts, aoi in [
    ("Fergana",  fc_ferg, aoi_ferg),
    ("Orenburg", fc_oren, aoi_oren),
]:
    for i, dates_chunk in enumerate(chunk(dates_all, DATES_PER_EXPORT), start=1):
        fc_chunk = build_fc_for_dates(fc_pts, aoi, dates_chunk, region_name)
        desc  = f"S2_{region_name}_chunk_{i:03d}"
        fname = f"{FILENAME_PREFIX}_{region_name}_chunk_{i:03d}"

        # Starts immediately inside the helper
        export_fc_to_drive(fc_chunk, desc, fname, DRIVE_FOLDER, SELECTORS)

        # Find the just-started task by description and poll it
        task = next((t for t in ee.batch.Task.list() if t.config.get('description') == desc), None)
        if task is None:
            print(f"[WARN] Could not find task just started: {desc}")
            continue

        while task.active():
            print(f"[{desc}] running...")
            time.sleep(30)

        print(f"[{desc}] state:", task.status().get('state'))

print("\n[INFO] S2 export tasks submitted.")
print("Tips:")
print(" - If tasks fail on memory/time, reduce DATES_PER_EXPORT (e.g., 5).")
print(" - If coverage is low (too many NaNs), try PAD_S2 = 5 or 7, but keep it as small as you can.")

[INFO] Unique S2 dates from test: 687
[WARN] No CRS in /content/drive/MyDrive/4th_place_GEO_AI_Cropland_Mapping_Challenge/raw_train_data/Fergana_training_samples.shp. Assuming EPSG:4326.
[WARN] No CRS in /content/drive/MyDrive/4th_place_GEO_AI_Cropland_Mapping_Challenge/raw_train_data/Orenburg_training_samples.shp. Assuming EPSG:4326.
[TASK] Started: S2_Fergana_chunk_001 → Drive/EE_S2_Exports/train_S2_Fergana_chunk_001.csv
[S2_Fergana_chunk_001] running...
[S2_Fergana_chunk_001] running...
[S2_Fergana_chunk_001] state: COMPLETED
[TASK] Started: S2_Fergana_chunk_002 → Drive/EE_S2_Exports/train_S2_Fergana_chunk_002.csv
[S2_Fergana_chunk_002] running...
[S2_Fergana_chunk_002] state: COMPLETED
[TASK] Started: S2_Fergana_chunk_003 → Drive/EE_S2_Exports/train_S2_Fergana_chunk_003.csv
[S2_Fergana_chunk_003] running...
[S2_Fergana_chunk_003] running...
[S2_Fergana_chunk_003] state: COMPLETED
[TASK] Started: S2_Fergana_chunk_004 → Drive/EE_S2_Exports/train_S2_Fergana_chunk_004.csv
[S2_Fergana_c

In [ ]:
s2_data_time = time.time()

print("Submitted S2 data for download in : ", (s2_data_time-s1_data_time)/60 ,"minutes")

Submitted S2 data for download in :  144.05825208822887 minutes


In [ ]:
print("Total time taken : ",(s2_data_time-start)/60 ,"minutes")

Total time taken :  227.65543197393418 minutes
